In [1]:
import pandas as pd
import sqlite3
from sqlite3 import Error

## Preprocesamiento de datos

**Dataset: *title.basics***

In [2]:
# Cargar el dataset basics
basics = pd.read_csv('datasets/title.basics.tsv.gz', sep='\t', na_values='\\N', dtype=str, compression='gzip')

In [3]:
# Filtrar solo títulos de tipo 'movie'
basics = basics[basics['titleType'].isin(["movie"])]
basics = basics.drop(columns=["titleType"])

# Filtrar los que tienen "Animation" en la columna de géneros
basics = basics[basics['genres'].notna()]
titles = basics[basics['genres'].str.contains("Animation", na=False)].copy()

# Convertir columnas a formato numérico donde aplica
titles['startYear'] = pd.to_numeric(titles['startYear'], errors='coerce').astype('Int64')  
titles['runtimeMinutes'] = pd.to_numeric(titles['runtimeMinutes'], errors='coerce').astype('Int64')  

# Seleccionar columnas que usaremos
titles = titles[[
    'tconst', 'primaryTitle', 'originalTitle',
    'startYear', 'runtimeMinutes', 'genres', 'isAdult'
]]

# Extraer todos los géneros únicos
all_genres = titles['genres'].str.split(',').explode().unique()
all_genres = all_genres[all_genres != "Animation"]

# Tabla genres
genres = pd.DataFrame({
    'id': range(1, len(all_genres)+1),
    'genre_name': all_genres
})

In [4]:
# Crear una lista de todas las combinaciones título-género
lista_temp = []

for _, row in titles.iterrows():
    tconst = row['tconst']
    if pd.notna(row['genres']):
        for genre in row['genres'].split(','):
            if genre != "Animation":
                # Agregar a la lista de combinaciones título-género
                lista_temp.append({
                    'tconst': tconst,
                    'genre_id': genres[genres['genre_name'] == genre]['id'].values[0]
                })

# Eliminar columna géneros de titles
titles = titles.drop(columns=["genres"])

# Tabla title_genres
title_genres = pd.DataFrame(lista_temp)

**Dataset *title.ratings***

In [5]:
# Cargar ratings
ratings = pd.read_csv('datasets/title.ratings.tsv.gz', sep='\t', na_values='\\N', dtype=str, compression='gzip')

# Tabla titles
titles = titles.merge(ratings, on="tconst", how="left")

**Dataset *title.akas***

In [6]:
# Cargar archivo akas
akas = pd.read_csv('datasets/title.akas.tsv.gz', sep='\t', na_values='\\N', dtype=str, compression='gzip')
akas = akas.rename(columns={"titleId": "tconst"})

In [7]:
# Nos interesa solo la región y el idioma
akas_filtrado = akas[['tconst', 'region', 'language']].copy()

# Obtener la lista de tconst de películas animadas
tconst_animaciones = titles['tconst'].unique()

# Filtrar akas para mantener solo las animaciones
akas_animaciones = akas_filtrado[akas_filtrado['tconst'].isin(tconst_animaciones)].copy()

In [8]:
# Tabla de regiones
regions = pd.DataFrame({
    'region_code': akas_animaciones['region'].dropna().unique()
}).dropna()
regions['id'] = range(1, len(regions) + 1)
regions = regions[['id', 'region_code']]

In [9]:
# Tabla de idiomas
languages = pd.DataFrame({
    'language_code': akas_animaciones['language'].dropna().unique()
}).dropna()
languages['id'] = range(1, len(languages) + 1)
languages = languages[['id', 'language_code']]

In [10]:
# Relación título-regiones
title_regions = (
    akas_animaciones[akas_animaciones['region'].notna()]
    [['tconst', 'region']]
    .merge(regions, left_on='region', right_on='region_code')
    [['tconst', 'id']]
    .rename(columns={'id': 'region_id'})
    .drop_duplicates()
)

In [11]:
# Relación título-idiomas
title_languages = (
    akas_animaciones[akas_animaciones['language'].notna()]
    [['tconst', 'language']]
    .merge(languages, left_on='language', right_on='language_code')
    [['tconst', 'id']]
    .rename(columns={'id': 'language_id'})
    .drop_duplicates()
)

**Dataset *title.principals***

In [12]:
principals = pd.read_csv('datasets/title.principals.tsv.gz', sep='\t', na_values='\\N', dtype=str, compression='gzip')

In [13]:
# Nos quedamos con directores, productores y escritores
principals_filtrado = principals[
    principals['category'].isin(['director', 'writer', 'producer'])
][['tconst', 'nconst', 'category']]

# Tabla title_roles
title_roles = principals_filtrado[principals_filtrado['tconst'].isin(tconst_animaciones)].copy()


**Dataset *name.basics***

In [14]:
names = pd.read_csv('datasets/name.basics.tsv.gz', sep='\t', na_values='\\N', dtype=str, compression='gzip')

In [15]:
# Obtener todas las personas únicas que participaron en animaciones
personas_animaciones = title_roles['nconst'].unique()

# Nos quedamos con identificador, nombre y profesiones
names_filtrado = names[['nconst', 'primaryName', 'knownForTitles']].copy()
names_filtrado = names_filtrado[names_filtrado['nconst'].isin(personas_animaciones)].copy()

In [16]:
# Tabla people
people = (
    names_filtrado[['nconst', 'primaryName']]
    .drop_duplicates(subset=['nconst'])
)

In [17]:
# Procesar knownForTitles para personas que trabajaron en animaciones
known_for = (names_filtrado[['nconst', 'knownForTitles']].dropna())

# Explotar la lista de títulos conocidos
known_for = (
    known_for.assign(knownForTitles=known_for['knownForTitles'].str.split(','))
    .explode('knownForTitles')
    .rename(columns={'knownForTitles': 'tconst'})
)

# Filtrar solo títulos que existen en nuestra base de animaciones
known_for = known_for[known_for['tconst'].isin(tconst_animaciones)]

# Tabla known_for
known_for = known_for.drop_duplicates().reset_index(drop=True)

## Cargando los datos
(no volver a ejecutar)

In [18]:
def create_connection(db_file):
    """Crear una conexión a la base de datos SQLite"""
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        print("Conexión exitosa a SQLite")
        return conn
    except Error as e:
        print(e)
    return conn

def load_dataframe_to_sqlite(conn, dataframe, table_name):
    """Cargar un DataFrame a una tabla SQLite"""
    try:
        dataframe.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Datos cargados exitosamente en la tabla {table_name}")
    except Error as e:
        print(f"Error al cargar datos en {table_name}: {e}")

def check_table_contents(conn, table_name, limit=5):
    """Mostrar algunas filas de una tabla para verificar"""
    try:
        query = f"SELECT * FROM {table_name} LIMIT {limit}"
        df = pd.read_sql(query, conn)
        print(f"\nContenido de la tabla {table_name}:")
        print(df.head(limit))
    except Error as e:
        print(f"Error al leer {table_name}: {e}")

In [19]:
database = "im.db"
conn = create_connection(database)

Conexión exitosa a SQLite


In [20]:
load_dataframe_to_sqlite(conn, regions, "regions")
check_table_contents(conn, "regions", 10)

Datos cargados exitosamente en la tabla regions

Contenido de la tabla regions:
   id region_code
0   1          CA
1   2         XWW
2   3          HU
3   4          AR
4   5          UA
5   6          ES
6   7          IT
7   8          US
8   9          AU
9  10          DE


In [21]:
load_dataframe_to_sqlite(conn, languages, "languages")
check_table_contents(conn, "languages", 10)

Datos cargados exitosamente en la tabla languages

Contenido de la tabla languages:
   id language_code
0   1            en
1   2            cs
2   3            ru
3   4            sr
4   5            fr
5   6            ja
6   7            es
7   8           qbn
8   9            nl
9  10            hr


In [22]:
load_dataframe_to_sqlite(conn, people, "people")
check_table_contents(conn, "people", 10)

Datos cargados exitosamente en la tabla people

Contenido de la tabla people:
      nconst       primaryName
0  nm0000019  Federico Fellini
1  nm0000037        Gene Kelly
2  nm0000092       John Cleese
3  nm0000104  Antonio Banderas
4  nm0000108        Luc Besson
5  nm0000158         Tom Hanks
6  nm0000161       Salma Hayek
7  nm0000165        Ron Howard
8  nm0000175      Stephen King
9  nm0000184      George Lucas


In [23]:
load_dataframe_to_sqlite(conn, titles, "titles")
check_table_contents(conn, "titles", 10)

Datos cargados exitosamente en la tabla titles

Contenido de la tabla titles:
      tconst                     primaryTitle  \
0  tt0007646                       El apóstol   
1  tt0008840             El apache de Londres   
2  tt0009469               Outwitting the Hun   
3  tt0009619                Sin dejar rastros   
4  tt0014140            How Troy Was Collared   
5  tt0014304            Napoleon Not So Great   
6  tt0015532  The Adventures of Prince Achmed   
7  tt0015830                  Foam Sweet Foam   
8  tt0020825             Un discípulo de caco   
9  tt0021309             The Story of the Fox   

                      originalTitle  startYear  isAdult  averageRating  \
0                        El apóstol       1917        0            NaN   
1              El apache de Londres       1918        0            5.6   
2                Outwitting the Hun       1918        0            6.2   
3                 Sin dejar rastros       1918        0            6.5   
4           

In [24]:
load_dataframe_to_sqlite(conn, genres, "genres")
check_table_contents(conn, "genres", 10)

Datos cargados exitosamente en la tabla genres

Contenido de la tabla genres:
   id   genre_name
0   1       Comedy
1   2        Drama
2   3      History
3   4    Adventure
4   5       Family
5   6      Fantasy
6   7  Documentary
7   8      Musical
8   9      Romance
9  10        Music


In [25]:
load_dataframe_to_sqlite(conn, title_genres, "title_genres")
check_table_contents(conn, "title_genres", 10)

Datos cargados exitosamente en la tabla title_genres

Contenido de la tabla title_genres:
      tconst  genre_id
0  tt0007646         1
1  tt0007646         2
2  tt0009469         1
3  tt0014140         1
4  tt0014140         3
5  tt0014304         1
6  tt0014304         3
7  tt0015532         4
8  tt0015532         2
9  tt0015830         1


In [26]:
load_dataframe_to_sqlite(conn, title_regions, "title_regions")
check_table_contents(conn, "title_regions", 10)

Datos cargados exitosamente en la tabla title_regions

Contenido de la tabla title_regions:
      tconst  region_id
0  tt0007646          1
1  tt0007646          2
2  tt0007646          3
3  tt0007646          4
4  tt0007646          5
5  tt0008840          6
6  tt0009469          7
7  tt0009469          8
8  tt0009619          4
9  tt0009619          9


In [27]:
load_dataframe_to_sqlite(conn, title_languages, "title_languages")
check_table_contents(conn, "title_languages", 10)

Datos cargados exitosamente en la tabla title_languages

Contenido de la tabla title_languages:
      tconst  language_id
0  tt0007646            1
1  tt0009619            1
2  tt0015532            2
3  tt0015532            1
4  tt0015532            3
5  tt0021309            4
6  tt0021309            3
7  tt0021309            1
8  tt0026793            3
9  tt0026793            5


In [28]:
load_dataframe_to_sqlite(conn, title_roles, "title_roles")
check_table_contents(conn, "title_roles", 10)

Datos cargados exitosamente en la tabla title_roles

Contenido de la tabla title_roles:
      tconst     nconst  category
0  tt0007646  nm0188105  director
1  tt0007646  nm0188105    writer
2  tt0007646  nm0884904  producer
3  tt0008840  nm0813682  director
4  tt0008840  nm0284960    writer
5  tt0008840  nm0813682    writer
6  tt0009469  nm0665163  director
7  tt0009469  nm0665163  producer
8  tt0009469  nm0712403  producer
9  tt0009619  nm0188105  director


In [29]:
load_dataframe_to_sqlite(conn, known_for, "known_for")
check_table_contents(conn, "known_for", 10)

Datos cargados exitosamente en la tabla known_for

Contenido de la tabla known_for:
      nconst     tconst
0  nm0000318  tt1142977
1  nm0000318  tt0121164
2  nm0000370  tt0029583
3  nm0000370  tt0048280
4  nm0000370  tt0046183
5  nm0000370  tt0034492
6  nm0000500  tt0243017
7  nm0000632  tt0389790
8  nm0000812  tt0442933
9  nm0000835  tt0076929


In [30]:
conn.commit()
conn.close()